In [3]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# --- Setup Project Paths ---
# Find the project root by looking for the 'data:clean:' directory
CWD = Path.cwd()
if (CWD / 'data:clean:').exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / 'data:clean:').exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError('Could not locate data:clean: folder from current working directory. Please run the notebook from the project root or the Code directory.')

CLEAN_DIR = PROJECT_ROOT / 'data:clean:'
all_files = sorted([f for f in CLEAN_DIR.glob('*.csv')])

print(f"Found {len(all_files)} CSV files in {CLEAN_DIR}")

Found 12 CSV files in /Users/timothytran/Documents/Python/clone/ecc3479-project/data:clean:


In [4]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm
import multiprocessing

# Set the start method for multiprocessing
# This is often necessary on macOS to avoid issues with notebooks
try:
    multiprocessing.set_start_method('fork', force=True)
except RuntimeError:
    # This will raise a RuntimeError if the context has already been set.
    # We can safely ignore it if that's the case.
    pass

def analyze_file(file_path):
    """
    Analyzes a single CSV file to extract summary statistics and correlation.
    """
    df = pd.read_csv(file_path)
    game_name = file_path.stem.replace('_', ' ').replace('-', ' ').title()
    
    # Add day and game columns
    df['day'] = range(len(df))
    df['game'] = game_name
    
    # Summary Statistics for 'Players' (z-scored)
    summary_stats = df['Players'].describe()
    
    # First-Order Correlation (Time Trend)
    r_day_players = df['day'].corr(df['Players'])
    
    # Add log-transformed players, handling zeros or negative values if any
    df['log_players'] = np.log(df['Players'].clip(lower=1))
    r_day_log_players = df['day'].corr(df['log_players'])
    
    result = {
        'Game': game_name,
        'Mean Players (Z-score)': summary_stats['mean'],
        'Std Dev Players': summary_stats['std'],
        'Min Players': summary_stats['min'],
        'Max Players': summary_stats['max'],
        'r(day, players)': r_day_players,
        'r(day, log(players))': r_day_log_players
    }
    
    return df, result

# --- Parallel Data Loading and Analysis ---
# It's good practice to wrap multiprocessing code in a __main__ check,
# especially in notebooks, to prevent issues during process spawning.
if __name__ == '__main__':
    all_data = []
    analysis_results = []

    # Use ProcessPoolExecutor to parallelize file processing
    with ProcessPoolExecutor() as executor:
        # Create a future for each file analysis
        futures = [executor.submit(analyze_file, fp) for fp in all_files]
        
        # Process results as they complete, with a progress bar
        for future in tqdm(as_completed(futures), total=len(all_files), desc="Analyzing game data"):
            df, result = future.result()
            all_data.append(df)
            analysis_results.append(result)

    # --- Display Results ---
    # Sort results by game name for consistent ordering
    analysis_results = sorted(analysis_results, key=lambda x: x['Game'])
    results_df = pd.DataFrame(analysis_results)

    print("Exploratory Analysis of Cleaned Game Data (Z-Scored Player Counts)")
    print(results_df.to_markdown(index=False))

    # --- Create a combined DataFrame for potential further analysis ---
    combined_df = pd.concat(all_data, ignore_index=True).sort_values(by=['game', 'day']).reset_index(drop=True)
    print("\nCombined DataFrame created with all game data.")

Analyzing game data:   0%|          | 0/12 [00:00<?, ?it/s]

Exploratory Analysis of Cleaned Game Data (Z-Scored Player Counts)
| Game                                           |   Mean Players (Z-score) |   Std Dev Players |   Min Players |   Max Players |   r(day, players) |   r(day, log(players)) |
|:-----------------------------------------------|-------------------------:|------------------:|--------------:|--------------:|------------------:|-----------------------:|
| Apex Legends 2025 01 28 To 2025 03 04          |              3.45403e-16 |                 1 |      -1.81793 |       1.70946 |         0.539019  |            -0.00160659 |
| Counter Strike 2 2024 04 23 To 2024 05 28      |              1.28485e-15 |                 1 |      -2.90128 |       1.71889 |         0.03115   |             0.010747   |
| Dead By Daylight 2023 11 14 To 2023 12 19      |              4.37921e-16 |                 1 |      -1.17071 |       3.12362 |         0.0269992 |            -0.0518734  |
| Deep Rock Galactic 2024 05 30 To 2024 07 04    |        

In [5]:
# --- Save the results to a markdown file ---
if __name__ == '__main__':
    output_path = PROJECT_ROOT / 'outputs:' / 'first_order_analysis_summary.md'
    output_path.parent.mkdir(exist_ok=True)
    
    markdown_output = results_df.to_markdown(index=False)
    
    with open(output_path, 'w') as f:
        f.write("# First-Order Analysis Summary\n\n")
        f.write("This table summarizes the player count data for each game, showing the mean, standard deviation, min, and max of the z-scored player counts over the observed period. It also includes the first-order correlation between the day and the player count, which indicates the overall trend.\n\n")
        f.write(markdown_output)

    print(f"\nAnalysis summary saved to: {output_path}")

# --- 5. Exploratory Analysis Of Variable Correlation ---

if __name__ == '__main__':
    # --- Pooled Correlation Analysis ---
    pooled_r_day_players = combined_df['day'].corr(combined_df['Players'])
    pooled_r_day_log_players = combined_df['day'].corr(combined_df['log_players'])
    
    # Calculate within-game z-score for players
    combined_df['players_z_within_game'] = combined_df.groupby('game')['Players'].transform(lambda x: (x - x.mean()) / x.std())
    pooled_r_day_z_players = combined_df['day'].corr(combined_df['players_z_within_game'])
    
    print("\n--- 5. Exploratory Analysis Of Variable Correlation ---\n")
    print("--- Pooled Correlation (across all games) ---")
    print(f"r(day, players) = {pooled_r_day_players:.4f}")
    print(f"r(day, log(players)) = {pooled_r_day_log_players:.4f}")
    print(f"r(day, z_within_game) = {pooled_r_day_z_players:.4f}\n")

    # --- Within-Game Trend Analysis ---
    positive_trend_count = (results_df['r(day, players)'] > 0).sum()
    strong_trends = results_df[results_df['r(day, players)'].abs() > 0.5].sort_values('r(day, players)', ascending=False)
    weak_trends = results_df[results_df['r(day, players)'].abs() < 0.2].sort_values('r(day, players)')

    print("--- Within-Game Trends ---")
    print(f"{positive_trend_count} of {len(results_df)} games have a positive r(day, players) trend.")
    
    print("\nExamples of stronger within-game first-order effects:")
    print(strong_trends[['Game', 'r(day, players)']].to_markdown(index=False))
    
    print("\nNear-flat/weak examples:")
    print(weak_trends[['Game', 'r(day, players)']].to_markdown(index=False))

    # --- Compact Per-Game Correlation Summary ---
    summary_table_df = results_df[['Game', 'r(day, players)', 'r(day, log(players))']].copy()
    summary_table_df['Direction'] = np.select(
        [summary_table_df['r(day, players)'] > 0.1, summary_table_df['r(day, players)'] < -0.1],
        ['Positive', 'Negative'],
        default='Flat'
    )
    
    print("\nCompact per-game correlation summary:")
    print(summary_table_df.to_markdown(index=False))
    
    # --- Save to Markdown File ---
    correlation_analysis_output_path = PROJECT_ROOT / 'outputs:' / 'variable_correlation_analysis.md'
    with open(correlation_analysis_output_path, 'w') as f:
        f.write("# 5. Exploratory Analysis Of Variable Correlation\n\n")
        f.write("## Pooled Correlation (across all games)\n")
        f.write(f"- r(day, players) = {pooled_r_day_players:.4f}\n")
        f.write(f"- r(day, log(players)) = {pooled_r_day_log_players:.4f}\n")
        f.write(f"- r(day, z_within_game) = {pooled_r_day_z_players:.4f}\n\n")
        f.write("## Within-Game Trends\n")
        f.write(f"{positive_trend_count} of {len(results_df)} games have a positive r(day, players) trend.\n\n")
        f.write("### Examples of stronger within-game first-order effects:\n")
        f.write(strong_trends[['Game', 'r(day, players)']].to_markdown(index=False) + "\n\n")
        f.write("### Near-flat/weak examples:\n")
        f.write(weak_trends[['Game', 'r(day, players)']].to_markdown(index=False) + "\n\n")
        f.write("## Compact per-game correlation summary:\n")
        f.write(summary_table_df.to_markdown(index=False))
        
    print(f"\nFull correlation analysis saved to: {correlation_analysis_output_path}")


Analysis summary saved to: /Users/timothytran/Documents/Python/clone/ecc3479-project/outputs:/first_order_analysis_summary.md

--- 5. Exploratory Analysis Of Variable Correlation ---

--- Pooled Correlation (across all games) ---
r(day, players) = 0.3455
r(day, log(players)) = -0.0193
r(day, z_within_game) = 0.3455

--- Within-Game Trends ---
10 of 12 games have a positive r(day, players) trend.

Examples of stronger within-game first-order effects:
| Game                                           |   r(day, players) |
|:-----------------------------------------------|------------------:|
| Palworld 2024 12 09 To 2025 01 13              |          0.785048 |
| No Man S Sky 2025 01 15 To 2025 02 19          |          0.771367 |
| Don T Starve Together 2023 04 13 To 2023 05 18 |          0.597799 |
| Deep Rock Galactic 2024 05 30 To 2024 07 04    |          0.560332 |
| Apex Legends 2025 01 28 To 2025 03 04          |          0.539019 |
| Warframe 2025 11 26 To 2025 12 31             